In [1]:
import sqlite3
import pandas as pd

# 1. Connect to the existing AcademyOps database
# We use '../' to go up one folder level from notebooks/ to reach data/
conn = sqlite3.connect('../data/academyops.db')

# 2. Extract the leads table into a DataFrame
df = pd.read_sql_query("SELECT * FROM leads", conn)

# 3. Close the connection
conn.close()

# 4. Display the first 5 rows to verify the extraction
df.head()

,id,name,phone,source,stage,notes,created_at,updated_at
0,114,Meera Nair,+917284649114,Google,New,Looking for career transition support,2026-04-21T14:38:18.764811,2026-04-21T14:38:18.764811
1,115,Gaurav Joshi,+918615238233,Google,New,Needs scholarship information,2026-05-21T14:38:18.764892,2026-05-21T14:38:18.764892
2,116,Vikram Trivedi,+917091463792,Google,New,"Self-taught, wants formal certification",2026-04-30T14:38:18.764911,2026-04-30T14:38:18.764911
3,117,Karan Kulkarni,+916729603372,Google,New,Working professional,2026-04-18T14:38:18.764922,2026-04-18T14:38:18.764922
4,118,Aditya Verma,+916690148110,Website,New,Recent graduate looking to upskill,2026-04-05T14:38:18.764932,2026-04-05T14:38:18.764932


In [2]:
# 1. Define our linear funnel order
stages = ['New', 'Contacted', 'Qualified', 'Demo', 'Enrolled']

# 2. Get the raw count of where leads are currently sitting
current_counts = df['stage'].value_counts().to_dict()

# 3. Calculate Cumulative Counts (Everyone in a stage + everyone past it)
cumulative_counts = {}
for i, stage in enumerate(stages):
    # Sum the current stage and all stages after it
    cumulative_counts[stage] = sum(current_counts.get(s, 0) for s in stages[i:])

print("--- Cumulative Funnel (Total Leads Reaching Each Stage) ---")
for stage, count in cumulative_counts.items():
    print(f"{stage}: {count}")

print("\n--- Stage-by-Stage Conversion Rates ---")
# 4. Calculate conversion rate from one stage to the next
for i in range(1, len(stages)):
    prev_stage = stages[i-1]
    curr_stage = stages[i]
    
    prev_count = cumulative_counts[prev_stage]
    curr_count = cumulative_counts[curr_stage]
    
    # Prevent division by zero
    rate = (curr_count / prev_count) * 100 if prev_count > 0 else 0
    print(f"{prev_stage} -> {curr_stage}: {rate:.1f}%")
    
# 5. Calculate Average Time to Reach Current Stage
print("\n--- Average Days to Reach Current Stage ---")
df['days_in_pipeline'] = (pd.to_datetime(df['updated_at']) - pd.to_datetime(df['created_at'])).dt.days
avg_time = df.groupby('stage')['days_in_pipeline'].mean().round(1)

# Reorder the output to match the funnel
for stage in stages:
    days = avg_time.get(stage, 0)
    print(f"{stage}: {days} days")

--- Cumulative Funnel (Total Leads Reaching Each Stage) ---
New: 115
Contacted: 55
Qualified: 27
Demo: 13
Enrolled: 5

--- Stage-by-Stage Conversion Rates ---
New -> Contacted: 47.8%
Contacted -> Qualified: 49.1%
Qualified -> Demo: 48.1%
Demo -> Enrolled: 38.5%

--- Average Days to Reach Current Stage ---
New: 0.0 days
Contacted: 2.0 days
Qualified: 11.9 days
Demo: 29.0 days
Enrolled: 45.6 days


In [3]:
from statsmodels.stats.proportion import proportions_ztest

# 1. Define what a "conversion" is (Reaching the 'Enrolled' stage)
df['converted'] = df['stage'].apply(lambda x: 1 if x == 'Enrolled' else 0)

# 2. Aggregate data by source to find totals and conversions
source_stats = df.groupby('source').agg(
    total_leads=('id', 'count'),  # Assuming 'id' is your primary key column
    conversions=('converted', 'sum')
).reset_index()

# Calculate the percentage
source_stats['conversion_rate'] = (source_stats['conversions'] / source_stats['total_leads']) * 100

print("--- Conversion Rates by Source ---")
print(source_stats.sort_values(by='conversion_rate', ascending=False).to_string(index=False))
print("\n")

# 3. Setup the Hypothesis Test
# Let's compare the two sources with the highest overall lead volume to ensure a robust sample size
top_sources = source_stats.sort_values(by='total_leads', ascending=False).head(2)
source_A = top_sources.iloc[0]
source_B = top_sources.iloc[1]

print(f"--- Hypothesis Test: {source_A['source']} vs {source_B['source']} ---")

# Isolate our conversion counts and total observations for the test
counts = [source_A['conversions'], source_B['conversions']]
nobs = [source_A['total_leads'], source_B['total_leads']]

# Run the Two-Proportion Z-Test
z_stat, p_value = proportions_ztest(counts, nobs)

print(f"Z-Statistic: {z_stat:.4f}")
print(f"P-Value: {p_value:.4f}")

# 4. Plain-Language Business Conclusion (Alpha = 0.05)
print("\n--- Formal Conclusion ---")
alpha = 0.05
if p_value < alpha:
    print(f"Because our p-value ({p_value:.4f}) is less than {alpha}, we REJECT the null hypothesis.")
    print(f"Result: There is a STATISTICALLY SIGNIFICANT difference in performance between {source_A['source']} and {source_B['source']}.")
else:
    print(f"Because our p-value ({p_value:.4f}) is greater than {alpha}, we FAIL TO REJECT the null hypothesis.")
    print(f"Result: There is NO statistically significant difference in performance between {source_A['source']} and {source_B['source']}. Any variance is likely due to random chance.")

--- Conversion Rates by Source ---
  source  total_leads  conversions  conversion_rate
Referral           27            2         7.407407
 Website           16            1         6.250000
Facebook           21            1         4.761905
  Google           22            1         4.545455
LinkedIn           29            0         0.000000


--- Hypothesis Test: LinkedIn vs Referral ---
Z-Statistic: -1.4926
P-Value: 0.1356

--- Formal Conclusion ---
Because our p-value (0.1356) is greater than 0.05, we FAIL TO REJECT the null hypothesis.
Result: There is NO statistically significant difference in performance between LinkedIn and Referral. Any variance is likely due to random chance.


In [4]:
import plotly.express as px
import pandas as pd

# --- 1. The Funnel Chart ---
# We use the cumulative_counts dictionary we created in Phase 2
funnel_data = pd.DataFrame({
    'Stage': list(cumulative_counts.keys()),
    'Leads': list(cumulative_counts.values())
})

fig_funnel = px.funnel(
    funnel_data, 
    x='Leads', 
    y='Stage', 
    title='AcademyOps Lead Conversion Funnel'
)
fig_funnel.show()

# --- 2. The Source Comparison Chart ---
# We use the source_stats DataFrame we created in Phase 3
fig_bar = px.bar(
    source_stats.sort_values('conversion_rate', ascending=False), 
    x='source', 
    y='conversion_rate', 
    title='Conversion Rate by Lead Source (%)',
    labels={'source': 'Lead Source', 'conversion_rate': 'Conversion Rate (%)'},
    text_auto='.1f' # This displays the exact percentage on top of the bars
)

# Optional: Add a distinct color to the bars for better presentation
fig_bar.update_traces(marker_color='rgb(55, 83, 109)')
fig_bar.show()

Executive Summary: Lead Funnel & Source Performance
1. Where do leads drop off the most?
The pipeline maintains a relatively stable conversion rate of roughly 48% to 49% through the early and middle stages (New → Contacted → Qualified → Demo). However, the most significant drop-off rate occurs at the very bottom of the funnel: Demo to Enrolled. Only 38.5% of leads who receive a demo actually enroll.

Additionally, time-in-stage reveals a major bottleneck. While leads are contacted quickly (averaging 2 days), it takes an average of 45.6 days for a lead to finally enroll, with the bulk of that time spent between the Qualified and Demo stages.

2. Which source converts the best?
Based on raw percentages, Referrals convert the best at a rate of 7.4% (2 conversions out of 27 leads). On the other end of the spectrum, LinkedIn is currently our lowest-performing source, driving the highest volume of leads (29) but yielding a 0% conversion rate.

3. Is this difference mathematically proven?
No. To ensure we do not make budget decisions based on statistical noise, we ran a Two-Proportion Z-Test comparing our top-performing source (Referral) against our highest-volume, lowest-performing source (LinkedIn).

Test Used: Two-Proportion Z-Test

Significance Level (Alpha): 0.05

P-Value: 0.1356

Conclusion: Because our p-value of 0.1356 is greater than our 0.05 threshold, we fail to reject the null hypothesis. There is no statistically significant difference in performance between LinkedIn and Referrals at this time. The current sample sizes and conversion counts are too small to mathematically prove that Referrals are inherently better than LinkedIn; the current variance could simply be due to random chance. Marketing should continue collecting data before making aggressive budget reallocations.